# 04 - Bayesian Lag Estimation

**Purpose:** Quantify uncertainty in the lag estimate using Bayesian inference.

**Key Question:** What is the posterior distribution of τ_lag (the intervention window)?

**Method:** Fit a PyMC model where food insecurity is predicted by a time-shifted energy price signal.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.ndimage import shift
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Aligned Data

In [ ]:
aligned = pl.read_parquet(PROCESSED_DATA / 'aligned_timeseries.parquet').to_pandas()
aligned['date'] = pd.to_datetime(aligned['date'])

print(f"Loaded {len(aligned)} months of aligned data")
print(f"Date range: {aligned['date'].min()} to {aligned['date'].max()}")
print(aligned.head())

## 2. Prepare Data for Bayesian Model

We model food insecurity as a linear function of lagged energy price:

$$\text{FoodInsecurity}_t = \alpha + \beta \cdot \text{EnergyPrice}_{t-\tau} + \epsilon_t$$

Where τ (tau) is the lag we want to estimate.

In [ ]:
# Normalize data for better model convergence
energy_raw = aligned['energy_price'].values
food_raw = aligned['food_insecurity'].values

# Standardize
energy_mean, energy_std = energy_raw.mean(), energy_raw.std()
food_mean, food_std = food_raw.mean(), food_raw.std()

energy = (energy_raw - energy_mean) / energy_std
food = (food_raw - food_mean) / food_std

n_obs = len(food)
print(f"Observations: {n_obs}")
print(f"Energy price: mean={energy_mean:.0f}, std={energy_std:.0f}")
print(f"Food insecurity: mean={food_mean:.1f}%, std={food_std:.1f}%")

## 3. Grid Search for Optimal Lag

Before Bayesian inference, let's do a quick grid search to confirm our cross-correlation finding.

In [ ]:
from scipy.stats import pearsonr

def compute_r2_at_lag(x, y, lag):
    """
    Compute R-squared between x shifted by lag and y.
    Positive lag means x is shifted forward (x leads y).
    """
    if lag > 0:
        x_shifted = x[:-lag]
        y_aligned = y[lag:]
    elif lag < 0:
        x_shifted = x[-lag:]
        y_aligned = y[:lag]
    else:
        x_shifted = x
        y_aligned = y
    
    corr, _ = pearsonr(x_shifted, y_aligned)
    return corr ** 2

# Grid search
lags = np.arange(0, 7)
r2_values = [compute_r2_at_lag(energy, food, lag) for lag in lags]

best_lag = lags[np.argmax(r2_values)]
best_r2 = max(r2_values)

print("Grid Search Results (Energy -> Food Insecurity):")
print("-" * 40)
for lag, r2 in zip(lags, r2_values):
    marker = " <-- BEST" if lag == best_lag else ""
    print(f"  Lag {lag} months: R² = {r2:.3f}{marker}")

print(f"\nOptimal lag: {best_lag} months (R² = {best_r2:.3f})")

## 4. Bayesian Model with PyMC

We'll fit a model that estimates the lag as a continuous parameter.

In [ ]:
try:
    import pymc as pm
    import arviz as az
    PYMC_AVAILABLE = True
    print(f"PyMC version: {pm.__version__}")
except ImportError:
    PYMC_AVAILABLE = False
    print("PyMC not available - using bootstrap estimation instead")

In [ ]:
if PYMC_AVAILABLE:
    # For discrete lag, we'll use a categorical approach
    # Testing lags from 0 to 6 months
    
    max_lag = 6
    
    # Pre-compute shifted predictors for each lag
    shifted_energy = {}
    for lag in range(max_lag + 1):
        if lag == 0:
            shifted_energy[lag] = energy
        else:
            # Pad with NaN and trim
            shifted = np.full_like(energy, np.nan)
            shifted[lag:] = energy[:-lag]
            shifted_energy[lag] = shifted
    
    # Use valid observations only (where all lags are defined)
    valid_idx = ~np.isnan(shifted_energy[max_lag])
    n_valid = valid_idx.sum()
    
    food_valid = food[valid_idx]
    energy_matrix = np.column_stack([shifted_energy[lag][valid_idx] for lag in range(max_lag + 1)])
    
    print(f"Valid observations for modeling: {n_valid}")
    print(f"Energy matrix shape: {energy_matrix.shape}")

In [ ]:
if PYMC_AVAILABLE:
    with pm.Model() as lag_model:
        # Prior on lag (discrete uniform from 0 to max_lag)
        lag_idx = pm.Categorical('lag_idx', p=np.ones(max_lag + 1) / (max_lag + 1))
        
        # Intercept and slope
        alpha = pm.Normal('alpha', mu=0, sigma=1)
        beta = pm.Normal('beta', mu=0, sigma=2)
        
        # Noise
        sigma = pm.HalfNormal('sigma', sigma=1)
        
        # Select the appropriate lagged predictor
        energy_lagged = pm.math.switch(
            pm.math.eq(lag_idx, 0), energy_matrix[:, 0],
            pm.math.switch(
                pm.math.eq(lag_idx, 1), energy_matrix[:, 1],
                pm.math.switch(
                    pm.math.eq(lag_idx, 2), energy_matrix[:, 2],
                    pm.math.switch(
                        pm.math.eq(lag_idx, 3), energy_matrix[:, 3],
                        pm.math.switch(
                            pm.math.eq(lag_idx, 4), energy_matrix[:, 4],
                            pm.math.switch(
                                pm.math.eq(lag_idx, 5), energy_matrix[:, 5],
                                energy_matrix[:, 6]
                            )
                        )
                    )
                )
            )
        )
        
        # Likelihood
        mu = alpha + beta * energy_lagged
        likelihood = pm.Normal('food', mu=mu, sigma=sigma, observed=food_valid)
        
        # Sample
        print("Sampling from posterior...")
        trace = pm.sample(2000, tune=1000, cores=1, random_seed=42, 
                          progressbar=True, return_inferencedata=True)
    
    print("\nSampling complete!")

## 5. Alternative: Bootstrap Confidence Interval

A simpler approach to quantify uncertainty using bootstrap resampling.

In [ ]:
def bootstrap_optimal_lag(energy, food, n_bootstrap=1000, max_lag=6, seed=42):
    """
    Bootstrap estimation of optimal lag with confidence interval.
    
    Uses block bootstrap to preserve temporal structure.
    """
    np.random.seed(seed)
    n = len(energy)
    block_size = 6  # ~6 month blocks
    
    optimal_lags = []
    
    for _ in range(n_bootstrap):
        # Block bootstrap
        n_blocks = n // block_size + 1
        block_starts = np.random.randint(0, n - block_size, size=n_blocks)
        
        indices = []
        for start in block_starts:
            indices.extend(range(start, min(start + block_size, n)))
        indices = np.array(indices[:n])
        
        energy_boot = energy[indices]
        food_boot = food[indices]
        
        # Find optimal lag for this bootstrap sample
        best_r2 = -1
        best_lag = 0
        for lag in range(max_lag + 1):
            try:
                r2 = compute_r2_at_lag(energy_boot, food_boot, lag)
                if r2 > best_r2:
                    best_r2 = r2
                    best_lag = lag
            except:
                pass
        
        optimal_lags.append(best_lag)
    
    return np.array(optimal_lags)

print("Running bootstrap estimation (1000 samples)...")
bootstrap_lags = bootstrap_optimal_lag(energy, food, n_bootstrap=1000)

# Summary statistics
lag_mean = bootstrap_lags.mean()
lag_median = np.median(bootstrap_lags)
lag_std = bootstrap_lags.std()
lag_ci_low = np.percentile(bootstrap_lags, 2.5)
lag_ci_high = np.percentile(bootstrap_lags, 97.5)

print(f"\nBootstrap Results:")
print(f"  Mean lag: {lag_mean:.1f} months")
print(f"  Median lag: {lag_median:.0f} months")
print(f"  Std dev: {lag_std:.1f} months")
print(f"  95% CI: [{lag_ci_low:.0f}, {lag_ci_high:.0f}] months")

In [ ]:
# Visualize bootstrap distribution
fig, ax = plt.subplots(figsize=(10, 5))

# Histogram
counts, bins, patches = ax.hist(bootstrap_lags, bins=np.arange(-0.5, 7.5, 1), 
                                 color='#2A9D8F', alpha=0.7, edgecolor='black')

# Highlight the mode
mode_lag = int(np.median(bootstrap_lags))
for i, patch in enumerate(patches):
    if bins[i] <= mode_lag < bins[i+1]:
        patch.set_facecolor('#E63946')

ax.axvline(x=lag_mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {lag_mean:.1f} months')
ax.axvline(x=lag_ci_low, color='gray', linestyle=':', linewidth=1.5, label=f'95% CI: [{lag_ci_low:.0f}, {lag_ci_high:.0f}]')
ax.axvline(x=lag_ci_high, color='gray', linestyle=':', linewidth=1.5)

ax.set_xlabel('Optimal Lag (months)', fontsize=12)
ax.set_ylabel('Bootstrap Frequency', fontsize=12)
ax.set_title('Posterior Distribution of Intervention Window\n(Bootstrap, n=1000)', fontsize=14)
ax.set_xticks(range(7))
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(OUTPUTS / 'lag_posterior.png', dpi=150)
plt.show()

print(f"\nSaved: lag_posterior.png")

## 6. Lag Distribution by Value

In [ ]:
# Count frequency of each lag
unique, counts = np.unique(bootstrap_lags, return_counts=True)
lag_probs = dict(zip(unique, counts / len(bootstrap_lags)))

print("Lag Probability Distribution:")
print("-" * 40)
for lag in range(7):
    prob = lag_probs.get(lag, 0)
    bar = '#' * int(prob * 50)
    print(f"  {lag} months: {prob*100:5.1f}% {bar}")

# Most likely lag
mode_lag = max(lag_probs, key=lag_probs.get)
mode_prob = lag_probs[mode_lag]
print(f"\nMost likely intervention window: {mode_lag} months ({mode_prob*100:.1f}% probability)")

## 7. Summary for Policy Report

In [ ]:
print("="*60)
print("BAYESIAN LAG ESTIMATION - SUMMARY")
print("="*60)
print(f"""
INDICATOR: Energy Price Cap -> Food Insecurity

POINT ESTIMATE:
  Optimal lag (grid search): {best_lag} months
  R-squared at optimal lag: {best_r2:.3f}

UNCERTAINTY QUANTIFICATION (Bootstrap, n=1000):
  Mean lag: {lag_mean:.1f} months
  Median lag: {lag_median:.0f} months
  Standard deviation: {lag_std:.1f} months
  95% Confidence Interval: [{lag_ci_low:.0f}, {lag_ci_high:.0f}] months

POLICY INTERPRETATION:
  When energy prices rise significantly, food insecurity follows
  with a delay of approximately {lag_median:.0f} months (95% CI: {lag_ci_low:.0f}-{lag_ci_high:.0f} months).
  
  This provides an intervention window of {lag_ci_low:.0f}-{lag_ci_high:.0f} months
  after an energy price shock is announced to deploy preventive measures.
""")

## 8. Save Results

In [ ]:
# Save detailed results
results = {
    'indicator': 'energy_price',
    'target': 'food_insecurity',
    'optimal_lag_months': int(best_lag),
    'r_squared': float(best_r2),
    'bootstrap_mean': float(lag_mean),
    'bootstrap_median': float(lag_median),
    'bootstrap_std': float(lag_std),
    'ci_95_low': float(lag_ci_low),
    'ci_95_high': float(lag_ci_high),
}

import json
with open(OUTPUTS / 'lag_estimation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Save bootstrap samples
np.save(OUTPUTS / 'bootstrap_lags.npy', bootstrap_lags)

print("Saved:")
print(f"  - {OUTPUTS / 'lag_estimation_results.json'}")
print(f"  - {OUTPUTS / 'bootstrap_lags.npy'}")
print(f"  - {OUTPUTS / 'lag_posterior.png'}")
print(f"\nNext: 05_kernel_fitting.ipynb or 06_synthesis.ipynb")